[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-deepmind/regress-lm/blob/main/colabs/kaggle_demo.ipynb)

# Kaggle Code Scoring with RegressLM

Fine-tune a pretrained RegressLM checkpoint on a new Kaggle competition and evaluate its ability to predict submission scores from code.

Recommended: H100-class GPU or better.

In [ ]:
!pip install -q "regress-lm[extras] @ git+https://github.com/google-deepmind/regress-lm.git"

In [ ]:
import math
import logging
import numpy as np
import torch
from regress_lm import core, rlm, tokenizers, vocabs
from regress_lm.pytorch import fine_tuning, optimizers

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Prerequisite: Hugging Face Token

The model uses a gated T5Gemma-2 checkpoint. Before continuing:
1. Accept the license agreement on the [`google/t5gemma-2-270m-270m`](https://huggingface.co/google/t5gemma-2-270m-270m) Hugging Face page.
2. Generate a "Read" token at [`huggingface.co/settings/tokens`](https://huggingface.co/settings/tokens).
3. Add the token to Colab's Secrets tab (key icon on the left) as `HF_TOKEN` and enable "Notebook access".

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

## 1. Load Pretrained Checkpoint

We load a RegressLM checkpoint pretrained on Kaggle code-scoring data. The model uses a [T5Gemma-2](https://blog.google/technology/developers/t5gemma-2/) 270M encoder with a 12-layer Transformer decoder.

In [ ]:
CHECKPOINT_URL = 'gs://regress-lm/checkpoints/kaggle/best_checkpoint.pt'  # TODO: update

decoder_vocab = vocabs.DecoderVocab(
    tokenizers.AddSpecialValues(
        tokenizers.IEEEFloatTokenizer(num_mantissa_digits=6),
        special_value_map={'NAN': math.nan},
    )
)

rlm_wrapper = rlm.RegressLM.from_t5gemma_encoder(
    model_name='google/t5gemma-2-270m-270m',
    decoder_vocab=decoder_vocab,
    max_input_len=8192,
    max_num_objs=1,
    num_decoder_layers=12,
)
model = rlm_wrapper.model
model.to(device)

# Load pretrained weights
import gcsfs
fs = gcsfs.GCSFileSystem()
with fs.open(CHECKPOINT_URL, 'rb') as f:
    state = torch.load(f, map_location='cpu')
model.load_state_dict(state['model_state'])
print(f'Loaded checkpoint. Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

## 2. Prepare Data

Each example is a Python code submission paired with its leaderboard score.
The input format is: `"{competition_id}\n{code}"`

In [ ]:
# --- Demo data ---
# In practice, replace with your own Kaggle competition data.
# Format: core.Example(x='<competition_id>\n<code>', y=<score>)

def make_demo_examples(n=500, comp_id='s4e4', seed=42):
    """Creates synthetic demo examples mimicking Kaggle code submissions."""
    rng = np.random.default_rng(seed)
    examples = []
    for i in range(n):
        imports = rng.choice(['import pandas as pd', 'import numpy as np', 'import sklearn'])
        model_type = rng.choice(['LinearRegression', 'RandomForest', 'GradientBoosting', 'XGBoost'])
        n_features = rng.integers(5, 50)
        code = f'{imports}\nfrom sklearn.model_selection import cross_val_score\n\ndef train(X, y):\n    model = {model_type}(n_features={n_features})\n    return cross_val_score(model, X, y, cv=5).mean()\n'
        base = {'LinearRegression': 0.3, 'RandomForest': 0.5, 'GradientBoosting': 0.6, 'XGBoost': 0.7}[model_type]
        score = base + rng.normal(0, 0.1)
        examples.append(core.Example(x=f'{comp_id}\n{code}', y=float(score)))
    return examples

all_examples = make_demo_examples(n=500)
print(f'Total examples: {len(all_examples)}')
print(f'Score range: [{min(e.y for e in all_examples):.3f}, {max(e.y for e in all_examples):.3f}]')
print(f'\nExample input (first 200 chars):\n{all_examples[0].x[:200]}')

## 3. Fine-Tuning

We fine-tune all model parameters on 256 examples with a held-out validation set for early stopping.

In [ ]:
from regress_lm.pytorch import fine_tuning, optimizers

SEED = 42
N_TRAIN = 256
N_VAL = 64

# Shuffle and split into train / val / test
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(all_examples))
train_examples = [all_examples[i] for i in perm[:N_TRAIN]]
val_examples = [all_examples[i] for i in perm[N_TRAIN:N_TRAIN + N_VAL]]
test_examples = [all_examples[i] for i in perm[N_TRAIN + N_VAL:]]

print(f'Train: {len(train_examples)}, Val: {len(val_examples)}, Test: {len(test_examples)}')

def optimizer_factory(named_params):
    trainable = [(n, p) for n, p in named_params if p.requires_grad]
    return optimizers.muon_adamw(trainable, lr=1e-5)

fine_tuner = fine_tuning.PyTorchFineTuner(
    model,
    optimizer_factory=optimizer_factory,
    max_epochs=10_000,
    batch_size=16,
    batch_size_per_device=4,  # fits on H100 80GB; use 8 for B200
    patience=10,
    use_lora=False,
)

fine_tuner.fine_tune(
    examples=train_examples,
    validation_examples=val_examples,
    seed=SEED,
)
print('Fine-tuning complete!')

## 4. Inference & Evaluation

We decode 128 samples per test example, then aggregate using the median.

In [ ]:
NUM_SAMPLES = 128
model.eval()

all_preds = []
with torch.inference_mode():
    for ex in test_examples:
        samples = rlm_wrapper.sample([core.ExampleInput(x=ex.x)], NUM_SAMPLES)
        all_preds.append(samples[0])  # (num_samples,)

preds = np.stack(all_preds)  # (N, num_samples)
y_true = np.array([ex.y for ex in test_examples])
y_pred = np.nanmedian(preds, axis=1)

print(f'Predictions shape: {preds.shape}')
print(f'NaN rate: {np.isnan(preds).mean():.1%}')

In [ ]:
from scipy import stats

valid = np.isfinite(y_true) & np.isfinite(y_pred)
y_t, y_p = y_true[valid], y_pred[valid]

print('=== Regression Metrics ===')
print(f'MSE:      {np.mean((y_t - y_p) ** 2):.4f}')
print(f'MAE:      {np.mean(np.abs(y_t - y_p)):.4f}')
print(f'Pearson:  {stats.pearsonr(y_t, y_p)[0]:.4f}')
print(f'Spearman: {stats.spearmanr(y_t, y_p)[0]:.4f}')
print(f'Kendall:  {stats.kendalltau(y_t, y_p)[0]:.4f}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_true, y_pred, alpha=0.6, s=20)
min_val, max_val = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5)
axes[0].set_xlabel('True Score'); axes[0].set_ylabel('Predicted Score (Median)')
axes[0].set_title(f'True vs Predicted (Pearson={stats.pearsonr(y_t, y_p)[0]:.3f})')

sorted_idx = np.argsort(y_true)
axes[1].fill_between(range(len(y_true)),
    np.nanpercentile(preds[sorted_idx], 25, axis=1),
    np.nanpercentile(preds[sorted_idx], 75, axis=1),
    alpha=0.3, label='25-75th percentile')
axes[1].plot(y_true[sorted_idx], 'k-', lw=1.5, label='True')
axes[1].plot(np.nanmedian(preds[sorted_idx], axis=1), 'r-', lw=1, label='Predicted')
axes[1].set_xlabel('Example (sorted by true score)')
axes[1].set_ylabel('Score'); axes[1].set_title('Predictions with Uncertainty')
axes[1].legend()

plt.tight_layout()
plt.show()